# Cell-level analysis (clustering, UMAP, endothelial cells) of GSE129363, GSE278526 and GSE268904: stage 1

**Author's note (junior bioinformatician):** this notebook is the first of two. It does the **cell-level** work on each
dataset on its own: quality control, dimension reduction, clustering, UMAP, naming the clusters and picking the
endothelial cells. Then it adds up the counts of the endothelial cells per person (pseudobulk) and tests T2D against
control. It writes one result table per dataset. The second notebook
(`GSE129363_GSE278526_combined_ERG.ipynb`) combines those tables.

## Why this order
Cells from one person are not independent, so the final statistics must be done per person. But the cells have to be
clustered first to know which ones are endothelial. So: **cells -> clusters -> endothelial cells -> per-person sums ->
test -> combine the results of the datasets.**

## The same dimension reduction for every dataset
The settings come from the code of the published adipose-tissue vascular atlas (AlZaim, Hassan and Kalucka, *Nature
Metabolism* 2026; github.com/Kalucka-lab/SAT_Atlas, notebooks `01` and `02`; the settings are listed in
`docs/paper_dimension_reduction_settings.md`):

1. log-normalise (scale factor 10,000), 2,000 variable genes, scale with the number of counts regressed out;
2. PCA (50 components), number of significant components by the authors' rule;
3. **Harmony** across samples (and across studies when several are put together);
4. all cells clustered at resolution **0.1**; the vascular clusters (endothelial, lymphatic, mural) are kept;
5. the whole reduction is repeated on the vascular cells: UMAP and neighbours on **30** Harmony components, clusters at
   resolution **0.7**;
6. clusters are named from marker genes. **PECAM1, CDH5 and the other genes we test are never used to name clusters**,
   so they cannot bias their own result.

Cells from SAT and VAT are clustered **together**; the depots are only separated at the final test.

## What it needs
The R packages `Seurat`, `harmony`, `Matrix`, `edgeR` and `ggplot2`. About 16 GB of memory for the largest dataset
(GSE268904, about 130,000 nuclei). Datasets whose path is left empty are skipped.

*Caution:* this code has been tested on small simulated data only, and without the real `harmony` package (a stand-in was
used). Check the cluster tables and the cell counts against what you expect before trusting any result.

## 1. Settings

**Cell annotation:** Choose the output folder and the paths of the three datasets. Leave a path as "" to skip that dataset.

In [ ]:
out_dir <- "cell_level_results"

# GSE129363: the folder that holds the Discovery Cohort files (matrix, genes, barcodes, CellAnnotation)
dir_129363  <- ""   # e.g. "C:/Users/user/Downloads/Jovita/Dataset/GSE129363"
# GSE278526: the archive and the cell metadata file (it has the donor of every cell)
tar_278526  <- ""   # e.g. ".../GSE278526/GSE278526_RAW.tar"
meta_278526 <- ""   # e.g. ".../GSE278526/GSE278526_cell_barcodes_metadata.gz"
# GSE268904: the raw archive
tar_268904  <- ""   # e.g. ".../GSE268904/GSE268904_RAW.tar"

work_dir <- "extracted_archives"   # where the .tar archives are unpacked

**Cell annotation:** Quality-control rules per dataset. GSE129363 uses the authors' own limits; GSE278526 uses the limits of its earlier notebook; GSE268904 is a nuclei dataset, so its mitochondrial limit is 5% (3-5% is typical for snRNA-seq) and it uses the atlas paper's upper limits.

In [ ]:
qc_rules <- list(
  GSE129363 = list(min_umi = 200, max_umi = 13750, min_genes = 201, max_genes = 2500, max_mito = 20),
  GSE278526 = list(min_umi = 0,   max_umi = Inf,   min_genes = 200, max_genes = Inf,  max_mito = 20),
  GSE268904 = list(min_umi = 500, max_umi = 60000, min_genes = 0,   max_genes = 10000, max_mito = 5))
mito_cutoffs_compare <- c(3, 5, 10, 15, 20)   # the QC table also shows how many cells would be left at these cut-offs

**Cell annotation:** Settings of the paper's pipeline, and the smallest number of endothelial cells a person needs to be included in the test.

In [ ]:
paper_res_all      <- 0.1   # clustering resolution for all cells (paper, notebook 01)
paper_res_vascular <- 0.7   # clustering resolution for the vascular cells (paper, notebook 02)
make_umaps         <- TRUE  # draw the UMAP of all cells (slow for 100,000+ cells; FALSE skips it)

min_endo_per_unit <- c(GSE129363 = 10, GSE278526 = 10, GSE268904 = 50)   # smallest number of endothelial cells per person and depot
covariates <- list(GSE129363 = character(0), GSE278526 = "sex", GSE268904 = character(0))   # extra terms in the test
use_geo_labels_129363 <- FALSE   # FALSE: diabetes labels of the annotation file; TRUE: GEO sample records (differs for 3 people)

panel <- c("ERG", "PECAM1", "CDH5", "NOS3", "VCAM1", "ICAM1", "HIF1A", "ANGPT2")   # the genes we test

# after looking at the cluster tables you can correct them here (a different list per dataset), then re-run
vascular_clusters_override <- list(GSE129363 = NULL, GSE278526 = NULL, GSE268904 = NULL)   # e.g. list(GSE268904 = c("0", "3"))
cluster_label_override     <- list(GSE129363 = NULL, GSE278526 = NULL, GSE268904 = NULL)   # e.g. list(GSE268904 = c("5" = "Venous"))

*What you should see:* nothing is printed.

## 2. Load the packages and make the output folder

**Cell annotation:** Load the packages. The notebook stops with a clear message if one is missing.

In [ ]:
need <- c("Seurat", "harmony", "Matrix", "edgeR", "ggplot2")
miss <- need[!vapply(need, requireNamespace, logical(1), quietly = TRUE)]
if (length(miss) > 0) stop("Please install: ", paste(miss, collapse = ", "), "  (install.packages / BiocManager::install)")
suppressPackageStartupMessages({library(Seurat); library(Matrix); library(edgeR); library(ggplot2)})
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)
dir.create(work_dir, showWarnings = FALSE, recursive = TRUE)
set.seed(1234)

## 3. The shared functions
These are used for every dataset, so all three get exactly the same treatment.

**Cell annotation:** Quality control. For every cell it works out the number of counts, genes and the percentage of mitochondrial reads, keeps cells inside the limits, and also records how many cells would remain at several mitochondrial cut-offs (for the QC comparison table).

In [ ]:
qc_filter <- function(m, gene, rule, name) {
  umi   <- Matrix::colSums(m)                                         # counts per cell
  nfeat <- Matrix::colSums(m > 0)                                     # genes detected per cell
  mito  <- Matrix::colSums(m[grepl("^MT-", gene), , drop = FALSE]) / pmax(umi, 1) * 100   # % mitochondrial reads
  ok_umi  <- umi >= rule$min_umi & umi <= rule$max_umi
  ok_gene <- nfeat >= rule$min_genes & nfeat <= rule$max_genes
  ok_mito <- is.na(rule$max_mito) | mito <= rule$max_mito
  keep <- ok_umi & ok_gene & ok_mito
  base <- ok_umi & ok_gene                                            # everything except the mitochondrial rule
  steps <- data.frame(dataset = name, step = c("cells in the data", "after count limits", "after gene limits",
                                               paste0("after mitochondrial limit (", rule$max_mito, "%)")),
                      cells = c(ncol(m), sum(ok_umi), sum(ok_umi & ok_gene), sum(keep)))
  comp <- data.frame(dataset = name, mito_cutoff_percent = mito_cutoffs_compare,
                     retained = sapply(mito_cutoffs_compare, function(cu) sum(base & mito <= cu)))
  comp$removed <- sum(base) - comp$retained
  comp$percent_lost <- round(100 * comp$removed / sum(base), 1)
  list(keep = keep, steps = steps, compare = comp)
}

*What you should see:* nothing is printed; the function is only defined here.

**Cell annotation:** The paper's rule for the number of significant principal components, and the paper's reduction (normalise, variable genes, scale, PCA, Harmony) as one function that is used twice per dataset.

In [ ]:
paper_sig_pcs <- function(stdev) {
  pct <- stdev / sum(stdev) * 100                                     # % variance of each component
  cum <- cumsum(pct)
  co1 <- which(cum > 90 & pct < 5)[1]
  co2 <- sort(which((pct[1:length(pct) - 1] - pct[2:length(pct)]) > 0.05), decreasing = TRUE)[1] + 1
  n <- min(co1, co2, na.rm = TRUE)
  if (!is.finite(n)) n <- 30                                          # fallback if neither rule gives a number
  n
}

paper_reduce <- function(obj) {
  DefaultAssay(obj) <- "RNA"
  obj <- NormalizeData(obj, normalization.method = "LogNormalize", scale.factor = 10000, verbose = FALSE)
  obj <- FindVariableFeatures(obj, selection.method = "vst", nfeatures = 2000, verbose = FALSE)
  obj <- ScaleData(obj, vars.to.regress = "nCount_RNA", verbose = FALSE)             # regress out the number of counts
  obj <- RunPCA(obj, features = VariableFeatures(obj), npcs = 50, verbose = FALSE)
  hv <- c("samples", if (length(unique(obj$Study_chemistry)) > 1) "Study_chemistry") # sample, plus study when several
  harmony::RunHarmony(obj, group.by.vars = hv, assay.use = "RNA", max.iter.harmony = 100, verbose = FALSE)
}

*What you should see:* nothing is printed; the functions are only defined here.

**Cell annotation:** Cluster naming. For each cluster it averages the log-normalised expression of every marker gene, centres each gene across clusters, averages per marker panel, and returns a clusters-by-panels score table.

In [ ]:
cluster_scores <- function(obj, panels, cluster_col) {
  genes <- intersect(unique(unlist(panels)), rownames(obj))
  dat <- GetAssayData(obj, assay = "RNA", layer = "data")[genes, , drop = FALSE]      # log-normalised values
  cl <- factor(obj@meta.data[[cluster_col]])
  mm <- Matrix::sparse.model.matrix(~ 0 + cl); colnames(mm) <- levels(cl)             # cells x clusters indicator
  avg <- as.matrix(dat %*% mm) %*% diag(1 / as.numeric(table(cl)), nlevels(cl))       # mean per cluster
  colnames(avg) <- levels(cl)
  z <- t(scale(t(avg))); z[!is.finite(z)] <- 0                                        # centre each gene across clusters
  sc <- sapply(panels, function(p) {
    p <- intersect(p, rownames(z))
    if (length(p) == 0) rep(NA_real_, ncol(z)) else colMeans(z[p, , drop = FALSE])    # mean z per panel
  })
  rownames(sc) <- colnames(z)
  sc
}

# marker panels (none contains one of the tested genes)
broad_panels <- list(
  endothelial = c("VWF", "CLDN5", "EGFL7", "FLT1", "PTPRB", "EMCN", "KDR"),
  lymphatic   = c("PROX1", "LYVE1", "CCL21", "TFF3", "MMRN1"),
  mural       = c("RGS5", "ACTA2", "MYH11", "PDGFRB", "NOTCH3"),
  fibroblast  = c("PDGFRA", "DCN", "LUM", "COL1A2", "COL3A1"),
  adipocyte   = c("ADIPOQ", "PLIN1", "LIPE", "LEP"),
  immune      = c("PTPRC", "CD68", "CD14", "C1QA", "CD3E"),
  mesothelial = c("MSLN", "UPK3B", "KRT19"))
vasc_panels <- list(
  endothelial = c("VWF", "CLDN5", "EGFL7", "FLT1", "PTPRB", "EMCN", "KDR"),
  arterial    = c("GJA5", "GJA4", "SEMA3G", "HEY1", "FBLN5", "SOX17", "EFNB2", "DLL4", "LTBP4", "BMX"),
  capillary   = c("CA4", "RGCC", "FABP4", "CD36", "RBP7", "GNG11", "BTNL9", "CD300LG", "APOLD1", "GPIHBP1"),
  venous      = c("ACKR1", "NR2F2", "SELP", "SELE", "PLVAP", "IL6", "CCL14", "TLL1", "ELOVL7"),
  lymphatic   = c("PROX1", "MMRN1", "PDPN", "CCL21", "TFF3", "LYVE1", "FLT4"),
  mural       = c("RGS5", "ACTA2", "MYH11", "PDGFRB", "NOTCH3", "TAGLN"),
  sub_AdEC    = c("PLIN1", "PDE3B", "ADH1B", "ACSL1", "ADIPOQ", "DCN", "LUM", "COL1A2", "PDGFRA"),
  immune      = c("PTPRC", "CD68", "LYZ", "CD3E", "C1QA"))
stopifnot(length(intersect(c(unlist(broad_panels), unlist(vasc_panels)), panel)) == 0)   # no tested gene in any panel

*What you should see:* nothing is printed. The last line stops the notebook if one of the eight tested genes ever gets into a marker list.

**Cell annotation:** The statistical test. For one set of people it fits edgeR (quasi-likelihood) on the summed counts: T2D relative to Control, with an optional extra term such as sex. The eight panel genes are always kept in the test, even if rare.

In [ ]:
edger_panel <- function(pb, info, covars) {
  info$group <- factor(info$group, levels = c("Control", "T2D"))
  terms <- "group"
  for (cv in covars) if (cv %in% names(info) && length(unique(info[[cv]])) > 1) {          # add a term only if it fits
    d <- model.matrix(as.formula(paste("~", paste(c(terms, cv), collapse = " + "))), info)
    if (qr(d)$rank == ncol(d) && nrow(d) - ncol(d) >= 2) terms <- c(terms, cv)
  }
  design <- model.matrix(as.formula(paste("~", paste(terms, collapse = " + "))), info)
  y <- DGEList(pb)
  keep <- filterByExpr(y, design)                                                           # drop genes with too few counts
  sel <- rownames(pb) %in% panel
  keep[sel] <- keep[sel] | rowSums(pb[sel, , drop = FALSE]) > 0                             # always keep the panel genes
  y <- y[keep, , keep.lib.sizes = FALSE]
  y <- if (exists("normLibSizes", where = asNamespace("edgeR"))) edgeR::normLibSizes(y) else edgeR::calcNormFactors(y)
  y <- estimateDisp(y, design, robust = TRUE)
  fit <- glmQLFit(y, design, robust = TRUE)
  topTags(glmQLFTest(fit, coef = "groupT2D"), n = Inf, sort.by = "none")$table
}

# runs the test for every cell group (subset) and every depot; returns the common result table
subset_tests <- function(cnt, cd, subsets, min_cells, covars, name) {
  rows <- list()
  for (sname in names(subsets)) for (dep in sort(unique(cd$depot))) {
    sel <- cd$depot == dep & cd$label %in% subsets[[sname]] & !is.na(cd$group)               # cells of this group and depot
    units <- unique(cd$unit[sel])                                                            # people (per depot) with such cells
    res <- NULL; n_ctl <- 0L; n_t2d <- 0L; tc <- setNames(rep(0, length(panel)), panel); status <- "not tested: no cells"
    if (length(units) > 0) {
      mm <- Matrix::sparse.model.matrix(~ 0 + factor(cd$unit[sel], levels = units)); colnames(mm) <- units
      pb <- as.matrix(cnt[, sel, drop = FALSE] %*% mm)                                       # add up counts per person
      info <- unique(cd[sel, c("unit", "donor", "group", covars), drop = FALSE])
      info <- info[match(units, info$unit), , drop = FALSE]
      ncell <- as.integer(table(factor(cd$unit[sel], levels = units)))
      use <- ncell >= min_cells                                                              # people with enough cells
      pb <- pb[, use, drop = FALSE]; info <- info[use, , drop = FALSE]
      n_ctl <- sum(info$group == "Control"); n_t2d <- sum(info$group == "T2D")
      tc <- rowSums(pb)[panel]; tc[is.na(tc)] <- 0
      status <- paste0("not tested: fewer than 2 people in a group (", n_ctl, " vs ", n_t2d, ")")
      if (min(n_ctl, n_t2d) >= 2) { res <- edger_panel(pb, info, covars); status <- "tested" }
    }
    i <- if (is.null(res)) rep(NA_integer_, length(panel)) else match(panel, rownames(res))
    rows[[length(rows) + 1]] <- data.frame(
      depot = dep, subset = sname, gene = panel, n_Control = n_ctl, n_T2D = n_t2d, total_counts = unname(tc[panel]),
      status = ifelse(is.na(i) & status == "tested", "not tested: filtered out", status),
      logFC = if (is.null(res)) NA_real_ else res$logFC[i], PValue = if (is.null(res)) NA_real_ else res$PValue[i],
      dataset = name, method = "edgeR pseudobulk", stringsAsFactors = FALSE)
  }
  do.call(rbind, rows)
}

*What you should see:* nothing is printed; the functions are only defined here.

**Cell annotation:** The main function: the paper's cell-level workflow for one dataset, from the cells to the result table. It runs steps 1 to 6 below and saves the tables and UMAPs.

In [ ]:
run_cell_level <- function(counts, meta, name) {
  cat("\n=====", name, ":", ncol(counts), "cells,", length(unique(meta$samples)), "samples =====\n")
  subsets <- list(All_blood_ECs = c("Arterial", "Capillary", "Venous", "Unassigned_AdEC"),   # broad endothelial population
                  Arterial = "Arterial", Capillary = "Capillary", Venous = "Venous", Lymphatic = "Lymphatic")
  savep <- function(p, file, w = 9, h = 7) ggsave(file.path(out_dir, paste0(name, "_", file, ".png")), p, width = w, height = h, dpi = 300, bg = "white")

  # 1. all cells: paper reduction, clusters, UMAP
  atlas <- CreateSeuratObject(counts = counts, meta.data = meta, min.cells = 0, min.features = 0)
  atlas <- paper_reduce(atlas)
  sdh <- Stdev(atlas, reduction = "harmony"); if (length(sdh) == 0) sdh <- apply(Embeddings(atlas, "harmony"), 2, sd)
  pcs <- 1:paper_sig_pcs(sdh)
  cat("Significant Harmony components (paper's rule):", length(pcs), "\n")
  atlas <- FindNeighbors(atlas, reduction = "harmony", dims = pcs, verbose = FALSE)
  atlas <- FindClusters(atlas, resolution = paper_res_all, verbose = FALSE)
  atlas$cluster_all <- as.character(atlas$seurat_clusters)
  if (make_umaps) {
    atlas <- RunUMAP(atlas, reduction = "harmony", dims = pcs, verbose = FALSE)
    options(repr.plot.width = 9, repr.plot.height = 7)
    for (v in c("cluster_all", "depot", "cohort")) {
      p <- DimPlot(atlas, group.by = v, label = (v == "cluster_all")) + ggtitle(paste(name, ": all cells by", v))
      print(p); savep(p, paste0("UMAP_all_cells_", v))
    }
  }

  # 2. name the clusters and keep the vascular ones
  sc <- cluster_scores(atlas, broad_panels, "cluster_all")
  btab <- data.frame(cluster = rownames(sc), n_cells = as.integer(table(atlas$cluster_all)[rownames(sc)]),
                     best_panel = colnames(sc)[apply(sc, 1, function(r) which.max(r))],
                     best_score = round(apply(sc, 1, max, na.rm = TRUE), 2), stringsAsFactors = FALSE)
  btab$vascular <- btab$best_panel %in% c("endothelial", "lymphatic", "mural")
  ov <- vascular_clusters_override[[name]]
  if (!is.null(ov)) btab$vascular <- btab$cluster %in% ov
  print(btab, row.names = FALSE)
  write.csv(btab, file.path(out_dir, paste0(name, "_all_cells_cluster_labels.csv")), row.names = FALSE)
  vcells <- colnames(atlas)[atlas$cluster_all %in% btab$cluster[btab$vascular]]
  cat("Vascular cells:", length(vcells), "\n")
  if (length(vcells) < 200) stop(name, ": fewer than 200 vascular cells; check the cluster table (vascular_clusters_override)")

  # 3. repeat the reduction on the vascular cells (30 Harmony components, resolution 0.7)
  vasc <- CreateSeuratObject(counts = LayerData(atlas, assay = "RNA", layer = "counts")[, vcells],
                             meta.data = atlas@meta.data[vcells, c("samples", "depot", "cohort", "Study_chemistry")],
                             min.cells = 0, min.features = 0)
  rm(atlas); invisible(gc())
  vasc <- paper_reduce(vasc)
  vasc <- RunUMAP(vasc, reduction = "harmony", dims = 1:30, verbose = FALSE)
  vasc <- FindNeighbors(vasc, reduction = "harmony", dims = 1:30, verbose = FALSE)
  vasc <- FindClusters(vasc, resolution = paper_res_vascular, verbose = FALSE)
  vasc$cluster_vasc <- as.character(vasc$seurat_clusters)

  # 4. name the vascular clusters (confident subtype needs a margin; unclear endothelial -> Unassigned_AdEC)
  sc <- cluster_scores(vasc, vasc_panels, "cluster_vasc")
  sub_cols <- setdiff(colnames(sc), "endothelial")
  top <- sub_cols[apply(sc[, sub_cols, drop = FALSE], 1, which.max)]
  srt <- t(apply(sc[, sub_cols, drop = FALSE], 1, sort, decreasing = TRUE))
  margin <- srt[, 1] - srt[, 2]; core <- sc[, "endothelial"]
  lab <- ifelse(top %in% c("lymphatic", "mural", "sub_AdEC", "immune") & margin >= 0.25, top,
         ifelse(top %in% c("arterial", "capillary", "venous") & margin >= 0.25 & core > 0, top,
         ifelse(core > 0, "Unassigned_AdEC", "other")))
  nice <- c(arterial = "Arterial", capillary = "Capillary", venous = "Venous", lymphatic = "Lymphatic",
            mural = "Mural", sub_AdEC = "Sub_AdEC", immune = "Immune")
  lab <- ifelse(lab %in% names(nice), nice[lab], lab)
  vtab <- data.frame(cluster = rownames(sc), n_cells = as.integer(table(vasc$cluster_vasc)[rownames(sc)]),
                     best_panel = top, margin = round(margin, 2), endothelial_core = round(core, 2), label = lab, stringsAsFactors = FALSE)
  ol <- cluster_label_override[[name]]
  if (!is.null(ol)) vtab$label[match(names(ol), vtab$cluster)] <- unname(ol)
  print(vtab, row.names = FALSE)
  write.csv(vtab, file.path(out_dir, paste0(name, "_vascular_cluster_labels.csv")), row.names = FALSE)
  vasc$label <- vtab$label[match(vasc$cluster_vasc, vtab$cluster)]
  p <- DimPlot(vasc, group.by = "label", label = TRUE, repel = TRUE) + ggtitle(paste(name, ": vascular cells, named clusters"))
  print(p); savep(p, "UMAP_vascular_labels")
  p <- DimPlot(vasc, group.by = "cohort") + ggtitle(paste(name, ": vascular cells by group")); print(p); savep(p, "UMAP_vascular_group")
  print(table(vasc$label, vasc$cohort))

  # 5. per-person sums of the raw counts and the test, per depot
  cd <- data.frame(cell = colnames(vasc), label = vasc$label, depot = vasc$depot,
                   samples = vasc$samples, meta[match(colnames(vasc), rownames(meta)), c("donor", "group", "unit", "sex")],
                   stringsAsFactors = FALSE)
  cnt <- LayerData(vasc, assay = "RNA", layer = "counts")
  res <- subset_tests(cnt, cd, subsets, min_endo_per_unit[[name]], covariates[[name]], name)
  write.csv(cd, file.path(out_dir, paste0(name, "_vascular_cells.csv")), row.names = FALSE)

  # 6. keep the endothelial cells (blood + lymphatic) for the optional integrated UMAP
  ec <- cd$label %in% c("Arterial", "Capillary", "Venous", "Unassigned_AdEC", "Lymphatic")
  list(results = res, endo_counts = cnt[, ec, drop = FALSE], endo_meta = cd[ec, ])
}

*What you should see:* nothing is printed; the function is only defined here. Everything it does is shown, step by step, when it runs below.

## 4. The data loaders
One function per dataset. Each returns a sparse count matrix of the cells that pass quality control, and a table with
one row per cell: sample, person, depot, group, and (where known) sex. Cells are not removed for any reason other than
quality control.

**Cell annotation:** GSE129363: read the Discovery Cohort files, keep the cells the authors annotated, and attach sample, person, depot and diabetes status from the annotation file.

In [ ]:
load_129363 <- function() {
  f <- list.files(dir_129363, recursive = TRUE, full.names = TRUE)
  find1 <- function(pat) { h <- f[grepl(pat, basename(f), ignore.case = TRUE)]; if (!length(h)) stop("GSE129363 file not found: ", pat); h[1] }
  rd <- function(p) if (grepl("\\.gz$", p)) gzfile(p) else p
  m <- as(readMM(rd(find1("Discovery_Cohort_matrix"))), "CsparseMatrix")                  # genes x cells
  gn <- read.delim(rd(find1("Discovery_Cohort_genes")), header = FALSE, stringsAsFactors = FALSE)
  bc <- read.delim(rd(find1("Discovery_Cohort_barcodes")), header = FALSE, stringsAsFactors = FALSE)[[1]]
  an <- read.delim(rd(find1("Discovery_Cohort_CellAnnotation")), stringsAsFactors = FALSE)
  gene <- make.unique(as.character(if (ncol(gn) >= 2) gn[[2]] else gn[[1]]))
  rownames(m) <- gene; colnames(m) <- bc
  m <- m[, an$CellID, drop = FALSE]                                                       # the annotated cells only

  sample <- as.character(an$SampleName); sample[sample == "Ate-10_VAT"] <- "Ate-10-VAT"  # one sample name is spelled differently
  donor  <- sub("[-_](SAT|VAT)$", "", sample)
  group  <- ifelse(an$Condition == "Diabetic", "T2D", "Control")
  if (use_geo_labels_129363) group[donor %in% c("SVF5", "Ate-04", "Ate-09")] <- "T2D"      # the three people the GEO records label diabetic
  meta <- data.frame(samples = sample, donor = donor, depot = an$Tissue, group = group, cohort = group,
                     unit = paste(an$Tissue, donor), sex = NA_character_, Study_chemistry = "GSE129363",
                     row.names = an$CellID, stringsAsFactors = FALSE)
  q <- qc_filter(m, gene, qc_rules$GSE129363, "GSE129363")
  list(counts = m[, q$keep], meta = meta[q$keep, ], qc = q$steps, compare = q$compare)
}

*What you should see:* nothing is printed; the function is only defined here.

**Cell annotation:** GSE278526: read the four libraries from the archive, keep the cells the metadata file calls single cells with a known person, and attach person, depot, group and sex.

In [ ]:
load_278526 <- function() {
  ex <- file.path(work_dir, "GSE278526")
  if (length(list.files(ex, pattern = "matrix", recursive = TRUE)) == 0) untar(tar_278526, exdir = ex)
  f <- list.files(ex, recursive = TRUE, full.names = TRUE)
  md <- read.delim(gzfile(meta_278526), stringsAsFactors = FALSE)
  md$participant <- trimws(md$participant)
  md <- md[md$status == "singlet" & !is.na(md$participant) & md$participant != "", ]          # single cells with a known person
  md$group <- unname(c(Healthy = "Control", Unhealthy = "T2D")[md$disease])
  md$depot <- unname(c(SQ = "SAT", Omentum = "VAT")[md$tissue])
  clean <- function(x) sub("-[0-9]+$", "", x)
  libs <- data.frame(gsm = c("GSM8548235", "GSM8548236", "GSM8548237", "GSM8548238"),
                     name = c("BRI-1456", "BRI-1457", "BRI-1458", "BRI-1459"), stringsAsFactors = FALSE)
  ms <- list(); mt <- list(); gene_ref <- NULL
  for (i in seq_len(nrow(libs))) {
    pick <- function(tag) f[grepl(libs$gsm[i], f) & grepl(tag, f)]
    rd <- function(p) if (grepl("\\.gz$", p)) gzfile(p) else p
    m <- as(readMM(rd(pick("matrix"))), "CsparseMatrix")
    ft <- read.delim(rd(pick("features|genes")), header = FALSE, stringsAsFactors = FALSE)
    bc <- read.delim(rd(pick("barcodes")), header = FALSE, stringsAsFactors = FALSE)[[1]]
    if (ncol(ft) >= 3) { g <- ft[[3]] == "Gene Expression"; m <- m[g, , drop = FALSE]; ft <- ft[g, , drop = FALSE] }
    gene <- make.unique(as.character(if (ncol(ft) >= 2) ft[[2]] else ft[[1]])); rownames(m) <- gene
    if (is.null(gene_ref)) gene_ref <- gene
    if (!identical(gene, gene_ref)) stop("Gene lists differ between libraries")
    sm <- md[md$sample_name == libs$name[i], ]
    pos <- match(clean(sm$barcode), clean(bc)); ok <- !is.na(pos)
    sm <- sm[ok, ]; m <- m[, pos[ok], drop = FALSE]
    colnames(m) <- paste(libs$name[i], sm$barcode, sep = "_")
    ms[[i]] <- m
    mt[[i]] <- data.frame(samples = libs$name[i], donor = sm$participant, depot = sm$depot, group = sm$group, cohort = sm$group,
                          unit = paste(sm$depot, sm$participant), sex = sm$sex, Study_chemistry = "GSE278526",
                          row.names = colnames(m), stringsAsFactors = FALSE)
  }
  m <- do.call(cbind, ms); meta <- do.call(rbind, mt)
  q <- qc_filter(m, gene_ref, qc_rules$GSE278526, "GSE278526")
  list(counts = m[, q$keep], meta = meta[q$keep, ], qc = q$steps, compare = q$compare)
}

*What you should see:* nothing is printed; the function is only defined here. In this dataset each library is one depot and one group, so Harmony by library also removes some of the group difference **in the UMAP only**. The test uses the raw counts, which Harmony never touches.

**Cell annotation:** GSE268904: read the 14 nucleus matrices, apply quality control to each, and put the nuclei together. The comparison is obese with T2D against obese without; lean people are used for clustering only.

In [ ]:
load_268904 <- function() {
  ex <- file.path(work_dir, "GSE268904")
  if (length(list.files(ex, pattern = "matrix", recursive = TRUE)) == 0) untar(tar_268904, exdir = ex)
  f <- list.files(ex, recursive = TRUE, full.names = TRUE)
  smp <- data.frame(gsm = sprintf("GSM%d", 8302405:8302418),
    donor = c("9S", "10S", "13S", "17S", "SAT22", "SAT54", "SAT1", "SAT18", "SAT34", "SAT39", "SAT4", "SAT51", "SAT58", "SAT60"),
    cohort = c("obese", "obese diabetic", "lean", "obese diabetic", "obese diabetic", "lean", "obese", "obese", "obese",
               "obese", "obese", "obese", "lean", "lean"),
    sex = c("female", "female", "female", "female", "male", "female", "male", "female", "male", "male", "male", "female", "male", "male"),
    stringsAsFactors = FALSE)
  ms <- list(); mt <- list(); qs <- list(); cp <- list(); gene_ref <- NULL
  for (i in seq_len(nrow(smp))) {
    pick <- function(tag) f[grepl(smp$gsm[i], f) & grepl(tag, f)]
    rd <- function(p) if (grepl("\\.gz$", p)) gzfile(p) else p
    m <- as(readMM(rd(pick("matrix"))), "CsparseMatrix")
    ft <- read.delim(rd(pick("features|genes")), header = FALSE, stringsAsFactors = FALSE)
    bc <- read.delim(rd(pick("barcodes")), header = FALSE, stringsAsFactors = FALSE)[[1]]
    if (ncol(ft) >= 3) { g <- ft[[3]] == "Gene Expression"; m <- m[g, , drop = FALSE]; ft <- ft[g, , drop = FALSE] }
    gene <- make.unique(as.character(if (ncol(ft) >= 2) ft[[2]] else ft[[1]])); rownames(m) <- gene
    if (is.null(gene_ref)) gene_ref <- gene
    if (!identical(gene, gene_ref)) stop("Gene lists differ between samples (", smp$gsm[i], ")")
    q <- qc_filter(m, gene, qc_rules$GSE268904, "GSE268904")
    m <- m[, q$keep, drop = FALSE]; colnames(m) <- paste(smp$donor[i], bc[q$keep], sep = "_")
    q$steps$dataset <- smp$donor[i]; q$compare$dataset <- smp$donor[i]                  # QC tables per person
    qs[[i]] <- q$steps; cp[[i]] <- q$compare
    grp <- if (smp$cohort[i] == "obese diabetic") "T2D" else if (smp$cohort[i] == "obese") "Control" else NA_character_
    ms[[i]] <- m
    nn <- ncol(m)                                                         # one row per nucleus
    mt[[i]] <- data.frame(samples = rep(smp$donor[i], nn), donor = rep(smp$donor[i], nn), depot = rep("SAT", nn),
                          group = rep(grp, nn), cohort = rep(smp$cohort[i], nn), unit = rep(paste("SAT", smp$donor[i]), nn),
                          sex = rep(smp$sex[i], nn), Study_chemistry = rep("GSE268904", nn),
                          row.names = colnames(m), stringsAsFactors = FALSE)
    cat(sprintf("%-6s %-15s nuclei kept: %6d\n", smp$donor[i], smp$cohort[i], ncol(m)))
    rm(m); invisible(gc())
  }
  list(counts = do.call(cbind, ms), meta = do.call(rbind, mt), qc = do.call(rbind, qs), compare = do.call(rbind, cp))
}

*What you should see:* nothing is printed; the function is only defined here.

## 5. Run the cell-level analysis on each dataset
Each cell below runs one dataset (skipped if its path is empty). The tables are printed as they are made, and the files
are saved to the output folder: cluster tables, UMAP figures (300 dpi), the quality-control tables and the result table.

**Cell annotation:** Run GSE129363: quality control, clustering and UMAP on all cells, vascular re-analysis, naming, and the per-person test in SAT and VAT.

In [ ]:
if (nzchar(dir_129363)) {
  d <- load_129363()
  cat("\nQuality control:\n"); print(d$qc, row.names = FALSE)
  cat("\nCells left at different mitochondrial cut-offs:\n"); print(d$compare, row.names = FALSE)
  write.csv(d$qc, file.path(out_dir, "GSE129363_QC_steps.csv"), row.names = FALSE)
  write.csv(d$compare, file.path(out_dir, "GSE129363_QC_mito_cutoffs.csv"), row.names = FALSE)
  out_GSE129363 <- run_cell_level(d$counts, d$meta, "GSE129363")
  rm(d); invisible(gc())
  write.csv(out_GSE129363$results, file.path(out_dir, "Panel_DE_T2D_vs_Control_GSE129363.csv"), row.names = FALSE)
  cat("\nSaved: Panel_DE_T2D_vs_Control_GSE129363.csv\n")
} else cat("GSE129363 skipped (no path set)\n")

*What you should see:* the QC steps and the mitochondrial comparison table, then the clusters of all cells (a table and UMAPs), the vascular clusters with their labels, the number of cells per label and group, and the file saved. If the endothelial cluster has only a few hundred cells, the later tests will have few reads.

**Cell annotation:** Run GSE278526 the same way.

In [ ]:
if (nzchar(tar_278526) && nzchar(meta_278526)) {
  d <- load_278526()
  cat("\nQuality control:\n"); print(d$qc, row.names = FALSE)
  cat("\nCells left at different mitochondrial cut-offs:\n"); print(d$compare, row.names = FALSE)
  write.csv(d$qc, file.path(out_dir, "GSE278526_QC_steps.csv"), row.names = FALSE)
  write.csv(d$compare, file.path(out_dir, "GSE278526_QC_mito_cutoffs.csv"), row.names = FALSE)
  out_GSE278526 <- run_cell_level(d$counts, d$meta, "GSE278526")
  rm(d); invisible(gc())
  write.csv(out_GSE278526$results, file.path(out_dir, "Panel_DE_T2D_vs_Control_GSE278526.csv"), row.names = FALSE)
  cat("\nSaved: Panel_DE_T2D_vs_Control_GSE278526.csv\n")
} else cat("GSE278526 skipped (no path set)\n")

*What you should see:* the same tables and UMAPs as above. In the first UMAP the groups may look separated by library; that is the design of this dataset (one library per depot and group), not necessarily biology.

**Cell annotation:** Run GSE268904 the same way (nuclei, 5% mitochondrial limit, SAT only). This is the slowest and most memory-hungry cell.

In [ ]:
if (nzchar(tar_268904)) {
  d <- load_268904()
  cat("\nQuality control:\n"); print(d$qc, row.names = FALSE)
  cat("\nCells left at different mitochondrial cut-offs:\n"); print(d$compare, row.names = FALSE)
  write.csv(d$qc, file.path(out_dir, "GSE268904_QC_steps.csv"), row.names = FALSE)
  write.csv(d$compare, file.path(out_dir, "GSE268904_QC_mito_cutoffs.csv"), row.names = FALSE)
  out_GSE268904 <- run_cell_level(d$counts, d$meta, "GSE268904")
  rm(d); invisible(gc())
  write.csv(out_GSE268904$results, file.path(out_dir, "Panel_DE_T2D_vs_Control_GSE268904.csv"), row.names = FALSE)
  cat("\nSaved: Panel_DE_T2D_vs_Control_GSE268904.csv\n")
} else cat("GSE268904 skipped (no path set)\n")

*What you should see:* one line per person while reading, then the same tables as above. Lean people are used for clustering but not for the test.

## 6. (Optional) All endothelial cells of the datasets in one UMAP
This is the paper's own integration step (Harmony across samples **and** across studies), applied to the endothelial and
lymphatic cells of the datasets that were run. It is **for display only**: it shows whether the datasets overlap and where
ERG is expressed. The statistics never use it; they come from the per-person sums above.

**Cell annotation:** Put the endothelial cells of all datasets together (common genes only), run the paper's reduction with Harmony across samples and studies, and draw the UMAP by dataset, label, depot and group. Skipped if fewer than two datasets were run.

In [ ]:
outs <- Filter(Negate(is.null), list(GSE129363 = if (exists("out_GSE129363")) out_GSE129363,
                                       GSE278526 = if (exists("out_GSE278526")) out_GSE278526,
                                       GSE268904 = if (exists("out_GSE268904")) out_GSE268904))   # datasets that were run
if (length(outs) >= 2) {
  genes_common <- Reduce(intersect, lapply(outs, function(o) rownames(o$endo_counts)))     # genes present in all datasets
  cnt_all <- do.call(cbind, lapply(outs, function(o) o$endo_counts[genes_common, , drop = FALSE]))
  em <- do.call(rbind, lapply(outs, function(o) o$endo_meta))
  ds <- rep(names(outs), sapply(outs, function(o) nrow(o$endo_meta)))                       # dataset of every cell
  em$cell_id <- paste(ds, em$cell, sep = "_"); colnames(cnt_all) <- em$cell_id
  rownames(em) <- em$cell_id
  em$Study_chemistry <- ds; em$dataset <- ds; em$cohort <- em$group
  int <- CreateSeuratObject(counts = cnt_all, meta.data = em[, c("samples", "depot", "cohort", "Study_chemistry", "label", "dataset")],
                            min.cells = 0, min.features = 0)
  int <- paper_reduce(int)                                                       # Harmony across samples and across studies
  int <- RunUMAP(int, reduction = "harmony", dims = 1:30, verbose = FALSE)
  options(repr.plot.width = 9, repr.plot.height = 7)
  for (v in c("dataset", "label", "depot", "cohort")) {
    p <- DimPlot(int, group.by = v) + ggtitle(paste("Endothelial cells of all datasets by", v))
    print(p); ggsave(file.path(out_dir, paste0("Integrated_endothelial_UMAP_", v, ".png")), p, width = 9, height = 7, dpi = 300, bg = "white")
  }
  if ("ERG" %in% rownames(int)) {
    int <- NormalizeData(int, verbose = FALSE)
    p <- FeaturePlot(int, features = "ERG", order = TRUE) + ggtitle("ERG (descriptive only, no test)")
    print(p); ggsave(file.path(out_dir, "Integrated_endothelial_UMAP_ERG.png"), p, width = 8, height = 7, dpi = 300, bg = "white")
  }
} else cat("Fewer than two datasets were run: no integrated UMAP.\n")

*What you should see:* four UMAPs of the endothelial cells of all datasets (by dataset, label, depot and group) and an ERG map. If the datasets overlap in the first plot, the shared labels are comparable; if they sit apart, the dataset differences are large and only the per-person results should be compared.

## 7. Hand over to the combined notebook

**Cell annotation:** List the result tables that were written. These are the inputs of the second notebook.

In [ ]:
files <- list.files(out_dir, pattern = "^Panel_DE_T2D_vs_Control_.*\\.csv$", full.names = TRUE)
print(files)
cat("\nIn GSE129363_GSE278526_combined_ERG.ipynb, section 1, set:\n")
cat('  file_129363 <- "', file.path(out_dir, "Panel_DE_T2D_vs_Control_GSE129363.csv"), '"\n', sep = "")
cat('  file_278526 <- "', file.path(out_dir, "Panel_DE_T2D_vs_Control_GSE278526.csv"), '"\n', sep = "")
cat('  file_268904_common <- "', file.path(out_dir, "Panel_DE_T2D_vs_Control_GSE268904.csv"), '"\n', sep = "")
sessionInfo()

*What you should see:* the file names, the three lines to paste into the second notebook, and the software versions.

## 8. How to read the result, and its limits
* The clusters and cell labels are **our annotation** from marker genes, not the original authors' labels. Check the cluster
  tables; correct them with the override settings if needed.
* An endothelial cluster with no clear subtype is `Unassigned_AdEC`: it is kept in the broad endothelial group
  (`All_blood_ECs`) but not used for any subtype-specific result.
* The test is per person and depot. People with fewer endothelial cells than `min_endo_per_unit` are left out.
* GSE278526: each library is one depot and one group, so disease and library cannot be separated.
* GSE129363: the diabetes labels differ between the annotation file and GEO for three people; set `use_geo_labels_129363`
  to run the other label set and say which one you used.
* GSE268904: only 3 T2D people, nuclei not cells, subcutaneous only, lean people not used in the test.
* The resolutions 0.1 and 0.7 are the authors' choices on their own atlas and were not re-tuned for these data.